In [1]:
"""SESSION 39: matched leakage experiment on Image_Annotation_rebuilt (YOLO12n-seg).

Two conditions share the SAME five source-grouped folds and the SAME canonical
held-out images. "grouped" removes every image of a held-out source group from
training; "overlap" adds the held-out group's augmented siblings to training.
The canonical images and any pixel-identical copies are never used for
training, validation, or checkpoint selection in either condition. A fixed
schedule runs with patience 0 and the FINAL weights are scored, so there is no
checkpoint selection at all. The only difference is held-out siblings in
training; the paired mAP difference is the measured leakage effect.

Fully self-contained. Kaggle GPU T4 x2, Internet on. RUN_ARGS near the top;
resume a partial run with RUN_ARGS = ["--resume-from", "/kaggle/input/<prev>"]."""
from __future__ import annotations

import argparse
import contextlib
import csv
import gc
import io
import json
import os
import random
import re
import shutil
import signal
import subprocess
import sys
import tempfile
import time
import zipfile
from collections import Counter
from pathlib import Path

import numpy as np

# One-cell use on Kaggle: put options here instead of on a command line,
# e.g. RUN_ARGS = ["--smoke"] or RUN_ARGS = ["--resume-from", "/kaggle/input/<prev>"].
RUN_ARGS: list[str] = []

SESSION_NAME = "session39_matched_leakage"
DEFAULT_MODEL = "yolo12n-seg.yaml"
LABELS = {
    "grouped": "Source-grouped",
    "overlap": "Source-overlap-permitted",
}
CONDITIONS = ("grouped", "overlap")
PRIMARY = ("overlap", "grouped")

IMAGE_SUFFIXES = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
IMAGE_ANNOTATION_CLASSES = ["Nail_psoriasis", "SJS-TEN", "acne", "vitiligo"]
# Offline transform prefixes on the source stem mark augmented variants.
AUGMENT_PREFIX = re.compile(
    r"^(?:"
    r"Crop-(?:100450_100400|100450_0300|100350_100300|0350_100400|0350_0300)|"
    r"padd-(?:100100_100100|1000_00|0100_00|00_1000|00_0100)|"
    r"Resize-450_400|"
    r"flip-(?:-1|0|1)|"
    r"(?:hue|saturation)-(?:200|150|100|50)|"
    r"superpixels-(?:200|100|75|50|25)"
    r")"
)


def _log(msg: str) -> None:
    print(f"[39] {time.strftime('%H:%M:%S')} {msg}", flush=True)


def _writable_dir() -> Path:
    return Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path(tempfile.gettempdir())


def _self_script() -> Path:
    """A file holding this script, so workers can start it in another process."""
    candidate = globals().get("__file__")
    if candidate and Path(candidate).is_file():
        return Path(candidate).resolve()
    try:
        history = get_ipython().user_ns.get("In", [])  # type: ignore[name-defined]
    except Exception:
        history = []
    for text in reversed(history):
        if isinstance(text, str) and "def run_stage_collect(" in text and "def _self_script(" in text:
            target = _writable_dir() / "SESSION_39_runtime.py"
            target.write_text(text, encoding="utf-8")
            return target
    raise RuntimeError("Cannot recover this script's source. Run it as a file "
                       "(!python path.py) or paste the whole script into one cell.")


# ---------------------------------------------------------------- dataset

def _split_dirs(root: Path, split: str) -> tuple[Path, Path]:
    split_names = {
        "val": ("val", "valid", "validation"),
        "valid": ("valid", "val", "validation"),
        "validation": ("validation", "valid", "val"),
    }.get(split, (split,))
    for split_name in split_names:
        nested = (root / "images" / split_name, root / "labels" / split_name)
        if all(p.is_dir() for p in nested):
            return nested
        legacy = (root / split_name / "images", root / split_name / "labels")
        if all(p.is_dir() for p in legacy):
            return legacy
    return root / split / "images", root / split / "labels"


def _load_class_names(root: Path) -> list[str]:
    config_path = root / "data.yaml"
    if not config_path.is_file():
        raise FileNotFoundError(f"Missing {config_path}; provide a YOLO data.yaml.")
    import yaml
    document = yaml.safe_load(config_path.read_text(encoding="utf-8")) or {}
    names_value = document.get("names")
    if isinstance(names_value, dict):
        names_value = [names_value[k] for k in sorted(names_value, key=lambda v: int(v))]
    if (not isinstance(names_value, list) or not names_value
            or not all(isinstance(n, str) and n.strip() for n in names_value)):
        raise ValueError(f"{config_path} must define names as an ordered list or mapping.")
    declared = document.get("nc")
    if declared is not None and int(declared) != len(names_value):
        raise ValueError(f"{config_path} declares nc={declared} but lists "
                         f"{len(names_value)} names.")
    return [n.strip() for n in names_value]


def _is_dataset_root(root: Path) -> bool:
    if not (root / "data.yaml").is_file():
        return False
    return all(img.is_dir() and lbl.is_dir()
               for img, lbl in (_split_dirs(root, s) for s in ("train", "val")))


def _is_image_annotation_dataset(root: Path) -> bool:
    """Match Image_Annotation_rebuilt by its 4-class names in data.yaml."""
    yaml_path = root / "data.yaml"
    if not yaml_path.is_file():
        return False
    try:
        text = yaml_path.read_text(encoding="utf-8")
    except OSError:
        return False
    return all(
        re.search(rf"^\s*\d+\s*:\s*{re.escape(name)}\s*$", text, re.MULTILINE)
        or re.search(rf"^\s*-\s*{re.escape(name)}\s*$", text, re.MULTILINE)
        for name in IMAGE_ANNOTATION_CLASSES
    )


def _find_dataset_root(hint: str) -> Path:
    if hint != "auto":
        explicit = Path(hint).expanduser().resolve()
        if _is_dataset_root(explicit):
            print(f"  [data] explicit dataset: {explicit}", flush=True)
            return explicit
        if explicit.is_dir():
            matches = sorted({p.parent.resolve() for p in explicit.rglob("data.yaml")
                              if _is_dataset_root(p.parent)})
            if len(matches) == 1:
                print(f"  [data] found dataset under explicit directory: {matches[0]}")
                return matches[0]
            if len(matches) > 1:
                raise RuntimeError("Multiple datasets under --data-root; pass one directly:\n"
                                   + "\n".join(f"    - {m}" for m in matches))
        raise FileNotFoundError(f"--data-root is not a complete YOLO dataset: {explicit}")

    kaggle_input = Path("/kaggle/input")
    search_roots = [kaggle_input] if kaggle_input.is_dir() else [Path.cwd()]
    candidates = set()
    for search_root in search_roots:
        if search_root.is_dir():
            for yaml_path in search_root.rglob("data.yaml"):
                root = yaml_path.parent.resolve()
                if _is_dataset_root(root):
                    candidates.add(root)
    if not candidates:
        raise FileNotFoundError("No complete YOLO dataset found. Attach it in Kaggle "
                                "or pass --data-root.")
    preferred = [p for p in candidates if _is_image_annotation_dataset(p)]
    if not preferred:
        preferred = [p for p in candidates
                     if "".join(c for c in p.name.lower() if c.isalnum())
                     in {"skindisnet2annotation", "imageannotationrebuilt"}]
    if not preferred:
        preferred = [p for p in candidates
                     if (p / "manifest.csv").is_file() and
                     {"name", "split", "source", "augmented"} <=
                     set(next(csv.reader(open(p / "manifest.csv", newline="",
                                              encoding="utf-8")), []))]
    pool = preferred if preferred else list(candidates)
    if len(pool) != 1:
        raise RuntimeError("Multiple complete datasets found. Pass --data-root "
                           "explicitly:\n" + "\n".join(f"    - {p}" for p in sorted(pool)))
    print(f"  [data] selected dataset: {pool[0]}", flush=True)
    return pool[0]


def _parse_label(lbl_path: Path, names: list[str]) -> list[dict]:
    anns = []
    try:
        for line in lbl_path.read_text().splitlines():
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls_id = int(parts[0])
            if cls_id < 0 or cls_id >= len(names):
                continue
            coords = list(map(float, parts[1:]))
            if len(coords) >= 4 and len(coords) % 2 == 0:
                xs, ys = coords[0::2], coords[1::2]
                x1, y1, x2, y2 = min(xs), min(ys), max(xs), max(ys)
                anns.append({"class_id": cls_id, "class_name": names[cls_id],
                             "polygon": list(zip(xs, ys)),
                             "bbox_norm": [x1, y1, x2 - x1, y2 - y1]})
    except Exception:
        pass
    return anns


def _discover_dataset(root: Path, names: list[str]):
    splits = []
    for split in ("train", "val", "test"):
        img_dir, lbl_dir = _split_dirs(root, split)
        items = []
        if img_dir.is_dir() and lbl_dir.is_dir():
            for img_path in sorted(img_dir.iterdir()):
                if img_path.suffix.lower() not in IMAGE_SUFFIXES:
                    continue
                lbl_path = lbl_dir / (img_path.stem + ".txt")
                if not lbl_path.exists():
                    continue
                anns = _parse_label(lbl_path, names)
                if anns:
                    items.append({"image_path": str(img_path), "label_path": str(lbl_path),
                                  "annotations": anns,
                                  "split": "valid" if split == "val" else split})
        splits.append(items)
    return splits


def _apply_source_manifest(root: Path, items: list[dict]) -> dict | None:
    """Attach source IDs from a manifest.csv with a 'source' column."""
    source_manifest = root / "manifest.csv"
    v2_manifest = root / "metadata" / "manifest.csv"
    if source_manifest.is_file():
        with source_manifest.open(newline="", encoding="utf-8") as handle:
            reader = csv.DictReader(handle)
            if not reader.fieldnames or not {"name", "split", "source", "augmented"} <= set(reader.fieldnames):
                raise ValueError(f"{source_manifest} must contain name, split, source, augmented")
            lookup = {}
            for row in reader:
                split = row.get("split", "").strip().lower()
                split = "valid" if split in {"val", "validation"} else split
                key = (split, Path(row.get("name", "")).stem)
                if not key[1] or not row.get("source", "").strip() or key in lookup:
                    raise ValueError("manifest.csv is incomplete or ambiguous")
                lookup[key] = row["source"].strip()
        manifest_kind = "source manifest"
        item_key = lambda it: (it.get("split", ""), Path(it["image_path"]).stem)
    elif v2_manifest.is_file():
        with v2_manifest.open(newline="", encoding="utf-8") as handle:
            reader = csv.DictReader(handle)
            if not reader.fieldnames or not {"image_file", "split_group_id"} <= set(reader.fieldnames):
                raise ValueError(f"{v2_manifest} must contain image_file, split_group_id")
            lookup = {Path(r["image_file"]).name: r["split_group_id"]
                      for r in reader if r.get("image_file")}
        manifest_kind = "image group manifest"
        item_key = lambda it: Path(it["image_path"]).name
    else:
        return None
    missing = []
    for item in items:
        source = lookup.get(item_key(item))
        if not source:
            missing.append(item["image_path"])
        else:
            item["group_id"] = source
    if missing:
        raise ValueError(f"{manifest_kind} is missing {len(missing)} images; first: {missing[0]}")
    return {"manifest_type": manifest_kind,
            "source_group_count": len({i["group_id"] for i in items})}


def _apply_filename_groups(items: list[dict]) -> dict | None:
    """Group Roboflow ".rf." variants by source stem when no manifest exists."""
    if not items or not any(".rf." in Path(i["image_path"]).stem for i in items):
        return None
    groups: dict[str, list[dict]] = {}
    for item in items:
        raw_stem = Path(item["image_path"]).stem.split(".rf.", 1)[0]
        source = AUGMENT_PREFIX.sub("", raw_stem, count=1)
        item["group_id"] = source
        item["augmented_filename"] = bool(AUGMENT_PREFIX.match(raw_stem))
        item["validation_image"] = False
        groups.setdefault(source, []).append(item)

    def class_key(item: dict) -> tuple:
        return tuple(sorted({a["class_id"] for a in item["annotations"]}))

    conflicts = {g for g, members in groups.items()
                 if len({class_key(m) for m in members}) > 1}
    for item in items:
        item["excluded"] = item["group_id"] in conflicts
    kept = [i for i in items if not i["excluded"]]
    by_group: dict[str, list[dict]] = {}
    for item in kept:
        by_group.setdefault(item["group_id"], []).append(item)
    for members in by_group.values():
        canonical = [m for m in members if not m["augmented_filename"]]
        min(canonical or members, key=lambda m: m["image_path"])["validation_image"] = True
    return {
        "manifest_type": "filename-derived source groups (no manifest)",
        "source_group_count": len(by_group),
        "excluded_conflict_groups": len(conflicts),
        "cross_split_groups": sum(len({m["split"] for m in members}) > 1
                                  for members in by_group.values()),
        "kept_images": len(kept),
    }


def _item_classes(item: dict) -> list[int]:
    return [a["class_id"] for a in item.get("annotations", [])]


def load_dataset(data_root: str) -> dict:
    """Discover the dataset and return the images that enter the CV folds."""
    root = _find_dataset_root(data_root)
    names = _load_class_names(root)
    train_items, valid_items, test_items = _discover_dataset(root, names)
    all_items = train_items + valid_items + test_items
    if len(all_items) < 10:
        raise RuntimeError(f"Only {len(all_items)} labelled images under {root}")
    evaluation_note = None
    filename_groups = None
    source_manifest = _apply_source_manifest(root, all_items)
    if source_manifest:
        trainval = train_items + valid_items
        overlap = ({i["group_id"] for i in trainval} & {i["group_id"] for i in test_items})
        if overlap:
            raise RuntimeError(f"The manifest places {len(overlap)} source groups in "
                               "both train/validation and test splits.")
        print(f"  [data] {source_manifest['manifest_type']}: "
              f"{source_manifest['source_group_count']} source groups", flush=True)
    else:
        filename_groups = _apply_filename_groups(all_items)
        if filename_groups:
            kept = [i for i in all_items if not i.get("excluded")]
            cross = filename_groups["cross_split_groups"]
            if cross:
                trainval = kept
                evaluation_note = ("Received splits pooled into source-grouped CV: "
                                   "variants cross the original boundaries, so the "
                                   "received test split is not independent. Held-out "
                                   "images are one original-preferred image per group.")
            else:
                trainval = [i for i in kept if i["split"] != "test"]
                evaluation_note = ("Filename-derived groups keep variants in one fold; "
                                   "the received test split is excluded from CV.")
            print(f"  [data] filename-derived source groups: "
                  f"{filename_groups['source_group_count']} groups, "
                  f"{filename_groups['kept_images']}/{len(all_items)} images kept "
                  f"({filename_groups['excluded_conflict_groups']} groups excluded for "
                  f"class-set disagreement); {cross} groups span more than one split"
                  + (" -- all splits pooled for grouped CV" if cross
                     else " -- test split excluded from CV"), flush=True)
        else:
            trainval = train_items + valid_items
            print("  [data] no source manifest -- per-image stratified folds "
                  "(test split excluded from CV)", flush=True)
    if not trainval:
        raise RuntimeError(f"No labelled train/validation images under {root}")
    print(f"  [data] root={root}  classes={len(names)}  cv_images={len(trainval)}", flush=True)
    print(f"  [data] class dist: {dict(Counter(names[c] for it in trainval for c in _item_classes(it)))}")
    return {"root": root, "names": names, "trainval": trainval,
            "source_manifest": source_manifest, "filename_groups": filename_groups,
            "evaluation_note": evaluation_note}


# ---------------------------------------------------------------- folds

def make_folds(items: list[dict], num_classes: int, n_splits: int, seed: int):
    """Stratified folds, grouping manifest-linked variants when available."""
    if items and all("group_id" in item for item in items):
        groups: dict[str, list[int]] = {}
        for index, item in enumerate(items):
            groups.setdefault(item["group_id"], []).append(index)
        if len(groups) < n_splits:
            raise ValueError(f"{n_splits}-fold CV needs at least {n_splits} source groups")
        group_ids = sorted(groups)
        label_matrix = np.zeros((len(items), num_classes), dtype=np.int32)
        for index, item in enumerate(items):
            for class_id in _item_classes(item):
                if 0 <= class_id < num_classes:
                    label_matrix[index, class_id] = 1
        features = np.asarray([
            [int(label_matrix[groups[g], c].sum()) for c in range(num_classes)]
            + [len(groups[g])] for g in group_ids], dtype=float)
        target = np.maximum(features.sum(axis=0) / n_splits, 1.0)
        best_assignment, best_objective = None, float("inf")
        for trial in range(32):
            rng = random.Random(seed + trial)
            order = list(range(len(group_ids)))
            rng.shuffle(order)
            order.sort(key=lambda i: -float((features[i] / target).max()))
            counts = np.zeros((n_splits, num_classes + 1), dtype=float)
            assignment = {}
            for index in order:
                costs = []
                for fold_id in range(n_splits):
                    projected = counts.copy()
                    projected[fold_id] += features[index]
                    costs.append(float(((projected / target) ** 2).sum()))
                minimum = min(costs)
                choices = [f for f, c in enumerate(costs) if abs(c - minimum) < 1e-10]
                picked = rng.choice(choices)
                counts[picked] += features[index]
                assignment[group_ids[index]] = picked
            objective = int((counts[:, :num_classes] == 0).sum()) * 1e6 + float(
                (((counts - target) / target) ** 2).sum())
            if objective < best_objective:
                best_objective, best_assignment = objective, assignment
        folds = []
        required = set(range(num_classes))
        for fold_id in range(n_splits):
            train = [i for i in items if best_assignment[i["group_id"]] != fold_id]
            valid = [i for i in items if best_assignment[i["group_id"]] == fold_id
                     and i.get("validation_image", True)]
            if not train or not valid:
                raise ValueError(f"Fold {fold_id} is empty after source-group assignment")
            held = {i["group_id"] for i in items if best_assignment[i["group_id"]] == fold_id}
            if {i["group_id"] for i in train} & held:
                raise AssertionError("A source group crossed a CV fold boundary")
            if {i["group_id"] for i in valid} != held:
                raise AssertionError(f"Fold {fold_id} has groups without a validation image")
            missing = required - {c for i in train for c in _item_classes(i)}
            if missing:
                raise ValueError(f"Fold {fold_id} train lacks classes {missing}")
            folds.append({"fold": fold_id, "train": train, "valid": valid})
        return folds
    # No group information: per-image stratified assignment.
    rng = random.Random(seed)
    per_class = {c: [] for c in range(num_classes)}
    for item in items:
        for c in _item_classes(item):
            if 0 <= c < num_classes:
                per_class[c].append(item)
    for bucket in per_class.values():
        rng.shuffle(bucket)
    folds = [{"fold": f, "train": [], "valid": []} for f in range(n_splits)]
    for bucket in per_class.values():
        for j, item in enumerate(bucket):
            folds[j % n_splits]["valid"].append(item)
    for f in folds:
        held = {i["image_path"] for i in f["valid"]}
        f["valid"] = sorted(f["valid"], key=lambda i: i["image_path"])
        f["train"] = [i for i in items if i["image_path"] not in held]
    return folds


def fold_manifest(cv_data: dict, n_splits: int, seed: int) -> dict:
    """Frozen fold assignment: relative image paths and source group per fold."""
    import hashlib
    root = cv_data["root"]
    folds = make_folds(cv_data["trainval"], len(cv_data["names"]), n_splits, seed)

    def rel(item: dict) -> str:
        return Path(item["image_path"]).relative_to(root).as_posix()

    out_folds, group_of = [], {}
    for fold in folds:
        out_folds.append({"fold": fold["fold"],
                          "train": [rel(i) for i in fold["train"]],
                          "valid": [rel(i) for i in fold["valid"]]})
        for item in fold["train"] + fold["valid"]:
            group_of[rel(item)] = item.get("group_id", rel(item))
    signature = hashlib.sha256(json.dumps(
        [f["valid"] for f in out_folds], sort_keys=True).encode()).hexdigest()
    return {"schema": "s38-fold-manifest-v1", "seed": seed, "n_splits": n_splits,
            "class_names": cv_data["names"],
            "grouping": (cv_data["filename_groups"] or cv_data["source_manifest"]),
            "signature": signature, "folds": out_folds, "group_of": group_of}


def inner_split(train_items: list[dict], frac: float, seed: int, fold_id: int):
    """Hold out whole source groups from a fold's training set for model selection."""
    groups = sorted({it["group_id"] for it in train_items})
    rng = random.Random(seed * 1000 + fold_id)
    inner = set(rng.sample(groups, max(1, round(len(groups) * frac))))
    fit = [it for it in train_items if it["group_id"] not in inner]
    val = [it for it in train_items if it["group_id"] in inner]
    return fit, val


def _label(item: dict) -> int:
    classes = {a["class_id"] for a in item["annotations"]}
    if len(classes) != 1:
        raise ValueError(f"{item['image_path']} has {len(classes)} classes; the classifier "
                         "needs exactly one disease per image")
    return classes.pop()


def _pixel_hash(image_path: str) -> str:
    """Hash of decoded pixels, so re-saved copies count as identical."""
    import cv2
    import hashlib
    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Unreadable image {image_path}")
    return hashlib.sha256(img.tobytes()).hexdigest()[:16]


def build_plan(cv_data: dict, folds: list[dict], selected: list[int], work: Path,
               inner_frac: float, seed: int) -> dict:
    names = cv_data["names"]
    all_items = cv_data["trainval"]
    plan = {"names": names, "root": str(cv_data["root"]), "folds": {}}
    for fold_id in selected:
        fold = folds[fold_id]
        fit, val = inner_split(fold["train"], inner_frac, seed, fold_id)
        held_groups = {it["group_id"] for it in fold["valid"]}
        assert not held_groups & {it["group_id"] for it in fit + val}, "held-out group in training"
        assert not {it["group_id"] for it in fit} & {it["group_id"] for it in val}, "inner leak"
        missing = set(range(len(names))) - {_label(it) for it in val}
        if missing:
            _log(f"WARNING fold {fold_id}: inner validation lacks classes "
                 f"{[names[c] for c in missing]}")
        fdir = work / "folds" / f"fold_{fold_id}"
        fdir.mkdir(parents=True, exist_ok=True)
        fit_list = sorted(it["image_path"] for it in fit)
        (fdir / "train.txt").write_text("\n".join(fit_list) + "\n")
        (fdir / "val.txt").write_text("\n".join(it["image_path"] for it in val) + "\n")
        yaml_path = fdir / "data.yaml"   # JSON is valid YAML
        yaml_path.write_text(json.dumps({"train": str((fdir / "train.txt").resolve()),
                                         "val": str((fdir / "val.txt").resolve()),
                                         "nc": len(names), "names": names}, indent=1))
        canonical_paths = {it["image_path"] for it in fold["valid"]}
        canon_hashes = {_pixel_hash(p) for p in canonical_paths}
        held_paths = {it["image_path"] for it in fold["valid"]}
        dropped, kept = [], []
        seen = set(fit_list)
        for it in [i for i in all_items if i["group_id"] in held_groups]:
            if it["image_path"] in canonical_paths or _pixel_hash(it["image_path"]) in canon_hashes:
                dropped.append(it["image_path"])
            elif it["image_path"] not in seen:
                kept.append(it["image_path"])
                seen.add(it["image_path"])
        overlap_list = fit_list + sorted(kept)
        if held_paths & set(overlap_list):
            raise AssertionError(f"Fold {fold_id}: held-out image inside overlap training list")
        (fdir / "train_overlap.txt").write_text("\n".join(overlap_list) + "\n")
        overlap_yaml = fdir / "data_overlap.yaml"
        overlap_yaml.write_text(json.dumps({"train": str((fdir / "train_overlap.txt").resolve()),
                                            "val": str((fdir / "val.txt").resolve()),
                                            "nc": len(names), "names": names}, indent=1))
        plan["folds"][str(fold_id)] = {
            "det_yaml": str(yaml_path),
            "det_yaml_overlap": str(overlap_yaml),
            "overlap_siblings_added": len(kept),
            "overlap_canonical_dropped": dropped,
            "fit": [[it["image_path"], _label(it)] for it in fit],
            "val": [[it["image_path"], _label(it)] for it in val],
            "heldout": [[it["image_path"], _label(it)] for it in fold["valid"]],
        }
        _log(f"fold {fold_id}: fit={len(fit)} images (+{len(kept)} overlap siblings, "
             f"{len(dropped)} canonical/pixel-dup excluded), inner val={len(val)} images "
             f"({len({it['group_id'] for it in val})} groups), held-out={len(fold['valid'])}")
    return plan


def import_previous(src: Path, work: Path, signature: str) -> None:
    sigs = list(src.rglob("plan_signature.txt"))
    if not sigs:
        raise FileNotFoundError(f"No Session 39 state (plan_signature.txt) under {src}")
    old = sigs[0].parent
    if sigs[0].read_text().strip() != signature:
        raise RuntimeError(f"{src} was made with a different fold assignment; refusing to reuse it")
    copied = 0
    for sub in ("runs", "det", "predictions", "folds"):
        if not (old / sub).is_dir():
            continue
        for path in (old / sub).rglob("*"):
            target = work / sub / path.relative_to(old / sub)
            if path.is_file() and not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(path, target)
                copied += 1
    _log(f"resumed {copied} file(s) from {old}")


# ---------------------------------------------------------------- scheduler

def run_jobs(jobs: list[dict], gpus: list[int], deadline: float, logs: Path,
             start_margin: float = 600) -> bool:
    """Run jobs one per GPU. At the deadline, stop running jobs (they resume later)."""
    logs.mkdir(parents=True, exist_ok=True)
    pending = [j for j in jobs if not j["done"]()]
    running: dict[int, tuple] = {}
    last_status = time.time()
    while running or (pending and time.time() < deadline - start_margin):
        for gpu in gpus:
            if gpu not in running and pending and time.time() < deadline - start_margin:
                job = pending.pop(0)
                handle = open(logs / f"{job['name']}.log", "a")
                env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu), PYTHONUNBUFFERED="1")
                proc = subprocess.Popen(job["cmd"], stdout=handle, stderr=subprocess.STDOUT,
                                        env=env, start_new_session=True)
                running[gpu] = (job, proc, handle)
                _log(f"started {job['name']} on GPU {gpu}")
        time.sleep(15)
        for gpu, (job, proc, handle) in list(running.items()):
            code = proc.poll()
            if code is None:
                continue
            handle.close()
            del running[gpu]
            if code != 0 or not job["done"]():
                tail = (logs / f"{job['name']}.log").read_text(errors="replace").splitlines()[-40:]
                raise RuntimeError(f"{job['name']} failed (exit {code}). Log tail:\n" + "\n".join(tail))
            _log(f"finished {job['name']}")
        if time.time() - last_status > 900:
            last_status = time.time()
            for job, _, _ in running.values():
                _log(f"status {job['name']}: {job['status']()}")
        if time.time() >= deadline and running:
            _log("session budget reached; stopping running jobs, they resume next session")
            for job, proc, handle in running.values():
                with contextlib.suppress(ProcessLookupError):
                    os.killpg(proc.pid, signal.SIGTERM)
            end = time.time() + 120
            for job, proc, handle in running.values():
                try:
                    proc.wait(timeout=max(1, end - time.time()))
                except subprocess.TimeoutExpired:
                    with contextlib.suppress(ProcessLookupError):
                        os.killpg(proc.pid, signal.SIGKILL)
                handle.close()
            running.clear()
    return all(j["done"]() for j in jobs)


# ---------------------------------------------------------------- workers

def _epochs_done(run_dir: Path) -> int:
    """Distinct epochs in results.csv (a resumed run can repeat the interrupted epoch's row)."""
    csv_path = run_dir / "results.csv"
    if not csv_path.is_file():
        return 0
    with open(csv_path, newline="") as handle:
        rows = [{k.strip(): v for k, v in r.items()} for r in csv.DictReader(handle)]
    return len({r.get("epoch", "").strip() for r in rows if r.get("epoch", "").strip()})


def worker_det(run: dict, plan: dict, fold_id: int, work: Path) -> int:
    import cv2
    import pycocotools.mask as mask_util
    import torch
    from ultralytics import YOLO

    condition = run["condition"]
    fold = plan["folds"][str(fold_id)]
    run_dir = work / "runs" / f"fold_{fold_id}_{condition}"
    det_yaml = fold["det_yaml" if condition == "grouped" else "det_yaml_overlap"]
    trained = run_dir / "TRAIN_DONE"
    batch = run["batch"]
    device = 0 if torch.cuda.is_available() else "cpu"
    if not trained.exists():
        last = run_dir / "weights" / "last.pt"
        if last.exists():
            ckpt = torch.load(last, map_location="cpu", weights_only=False)
            if ckpt.get("epoch", -1) == -1 or ckpt.get("optimizer") is None:
                print(f"fold {fold_id} ({condition}) already trained; skipping to prediction",
                      flush=True)
            else:
                print(f"resuming fold {fold_id} ({condition}) from {last} "
                      f"({_epochs_done(run_dir)} epochs done)", flush=True)
                YOLO(str(last)).train(resume=True)
            del ckpt
        else:
            while True:
                try:
                    # yolo12n-seg.pt is not a downloadable asset; build from the
                    # yaml and transfer compatible layers from yolo12n.pt
                    # (downloadable), matching Session 35.
                    seg_model = YOLO(run["model"])
                    init = run.get("init_weights")
                    if init and init.lower() != "none":
                        seg_model.load(init)
                    seg_model.train(
                        # Fixed schedule, patience 0: no early stop, FINAL weights scored.
                        data=det_yaml, task="segment", epochs=run["epochs"],
                        imgsz=run["imgsz"], batch=batch, workers=run["workers"], device=device,
                        patience=0, seed=run["seed"] + fold_id,
                        project=str(work / "runs"), name=f"fold_{fold_id}_{condition}",
                        exist_ok=True, amp=True, plots=False, verbose=False, save_period=-1)
                    break
                except Exception as err:
                    if "out of memory" in str(err).lower() and batch > 2:
                        batch //= 2
                        print(f"out of memory; retrying fold {fold_id} ({condition}) "
                              f"with batch {batch}", flush=True)
                        shutil.rmtree(run_dir, ignore_errors=True)
                        gc.collect()
                        torch.cuda.empty_cache()
                        continue
                    raise
        trained.write_text(json.dumps({"batch": batch}))

    weights = run_dir / "weights" / "last.pt"
    if not weights.is_file():
        weights = run_dir / "weights" / "best.pt"
    model = YOLO(str(weights))
    anns: list[dict] = []
    offset = fold_id * 10_000
    for idx, (path, _) in enumerate(fold["heldout"]):
        image = cv2.imread(path)
        height, width = image.shape[:2]
        result = model.predict(source=path, conf=run["conf"], iou=run["iou"],
                               imgsz=run["imgsz"], max_det=run["det_max"], device=device,
                               verbose=False)[0]
        if result.boxes is None or len(result.boxes) == 0:
            continue
        polygons = list(result.masks.xy) if result.masks is not None else None
        for i, (box, score, cls) in enumerate(zip(result.boxes.xyxy.cpu().numpy(),
                                                  result.boxes.conf.cpu().numpy(),
                                                  result.boxes.cls.cpu().numpy().astype(int))):
            x1, y1, x2, y2 = (float(v) for v in box)
            mask = np.zeros((height, width), dtype=np.uint8)
            poly = polygons[i] if polygons is not None and i < len(polygons) else None
            if poly is not None and len(poly) >= 3:
                cv2.fillPoly(mask, [np.round(np.asarray(poly)).astype(np.int32).reshape(-1, 1, 2)], 1)
            rle = mask_util.encode(np.asfortranarray(mask))
            rle["counts"] = rle["counts"].decode("utf-8")
            anns.append({"id": offset * 1000 + len(anns), "image_id": offset + idx,
                         "file_name": Path(path).name, "category_id": int(cls) + 1,
                         "score": float(score), "bbox": [x1, y1, x2 - x1, y2 - y1],
                         "segmentation": rle, "area": float(mask_util.area(rle)),
                         "iscrowd": 0})
    out = work / "det" / condition
    out.mkdir(parents=True, exist_ok=True)
    (out / f"det_fold_{fold_id}_metrics.json").write_text(json.dumps({
        "fold": fold_id, "condition": condition, "model": run["model"],
        "init_weights": run.get("init_weights"),
        "epochs_requested": run["epochs"], "epochs_trained": _epochs_done(run_dir),
        "weights": weights.name, "overlap_siblings_added": fold["overlap_siblings_added"],
        "batch": json.loads(trained.read_text()).get("batch"), "detections": len(anns),
        "manifest_signature": run["signature"]}, indent=1))
    (out / f"det_fold_{fold_id}.json").write_text(json.dumps(anns))
    print(f"fold {fold_id} ({condition}): {len(anns)} detections saved", flush=True)
    return 0


# ---------------------------------------------------------------- collect

def build_gt_coco(items: list[dict], names: list[str], img_id_offset: int):
    """Held-out ground truth in COCO segm form; masks from the label polygons."""
    import cv2
    import pycocotools.mask as mask_util
    gt_images, gt_anns = [], []
    ann_id = img_id_offset * 10_000 + 1
    for rel_idx, item in enumerate(items):
        img = cv2.imread(item["image_path"])
        if img is None:
            continue
        height, width = img.shape[:2]
        img_id = img_id_offset + rel_idx
        gt_images.append({"id": img_id, "file_name": Path(item["image_path"]).name,
                          "width": width, "height": height})
        for ann in item["annotations"]:
            pts = np.asarray([[x * width, y * height] for x, y in ann["polygon"]],
                             dtype=np.int32).reshape(-1, 1, 2)
            mask = np.zeros((height, width), dtype=np.uint8)
            cv2.fillPoly(mask, [pts], 1)
            ys, xs = np.where(mask)
            if len(xs) == 0 or xs.max() - xs.min() < 2 or ys.max() - ys.min() < 2:
                continue
            rle = mask_util.encode(np.asfortranarray(mask))
            rle["counts"] = rle["counts"].decode("utf-8")
            gt_anns.append({"id": ann_id, "image_id": img_id,
                            "category_id": ann["class_id"] + 1, "iscrowd": 0,
                            "bbox": [float(xs.min()), float(ys.min()),
                                     float(xs.max() - xs.min()), float(ys.max() - ys.min())],
                            "segmentation": rle, "area": float(mask_util.area(rle))})
            ann_id += 1
    return gt_images, gt_anns


def run_stage_collect(run: dict, work: Path) -> int:
    """Assemble held-out ground truth and per-condition predictions for scoring."""
    cv_data = load_dataset(run["data_root"])
    names = cv_data["names"]
    folds = make_folds(cv_data["trainval"], len(names), 5, run["fold_seed"])
    if fold_manifest(cv_data, 5, run["fold_seed"])["signature"] != run["signature"]:
        raise RuntimeError("The dataset produces different folds than the plan")

    all_images, all_anns = [], []
    for fold_id in run["folds"]:
        items = folds[fold_id]["valid"]
        gt_images, gt_anns = build_gt_coco(items, names, fold_id * 10_000)
        all_images += gt_images
        all_anns += gt_anns
        _log(f"collect: fold {fold_id} GT done")
    id_of = {img["file_name"]: img["id"] for img in all_images}
    cats = [{"id": i + 1, "name": n} for i, n in enumerate(names)]
    pdir = work / "predictions"
    pdir.mkdir(parents=True, exist_ok=True)
    (pdir / "oof_gt.json").write_text(json.dumps({"info": {}, "licenses": [], "categories": cats,
                                                  "images": all_images, "annotations": all_anns}))
    for condition in run["conditions"]:
        preds = []
        for fold_id in run["folds"]:
            dets = json.loads((work / "det" / condition / f"det_fold_{fold_id}.json").read_text())
            for ann in dets:
                if ann["file_name"] not in id_of:
                    raise RuntimeError(f"Fold {fold_id} ({condition}): detections outside held-out")
                preds.append({**ann, "image_id": id_of[ann["file_name"]]})
            _log(f"collect: fold {fold_id} ({condition}) predictions done")
        (pdir / f"detector_{condition}.json").write_text(json.dumps(preds))
    return 0


# ---------------------------------------------------------------- scoring

def _coco_gt(images, annotations, categories):
    from pycocotools.coco import COCO
    coco = COCO()
    # Ids start at 1: COCOeval reads a matched ground-truth id of 0 as "unmatched".
    annotations = [{**a, "id": i + 1} for i, a in enumerate(annotations)]
    coco.dataset = {"info": {}, "licenses": [], "categories": categories,
                    "images": images, "annotations": annotations}
    with contextlib.redirect_stdout(io.StringIO()):
        coco.createIndex()
    return coco


def _box_iou(a, b) -> float:
    """IoU of two xywh boxes."""
    ax2, ay2, bx2, by2 = a[0] + a[2], a[1] + a[3], b[0] + b[2], b[1] + b[3]
    iw, ih = max(0.0, min(ax2, bx2) - max(a[0], b[0])), max(0.0, min(ay2, by2) - max(a[1], b[1]))
    inter = iw * ih
    union = a[2] * a[3] + b[2] * b[3] - inter
    return inter / union if union > 0 else 0.0


def _counts_at(images, annotations, preds, conf_thr: float, iou_thr: float):
    """Pooled TP/FP/FN: each prediction matches the best-IoU unmatched GT of its class."""
    gt_by_img = {}
    for ann in annotations:
        gt_by_img.setdefault(ann["image_id"], []).append(ann)
    pred_by_img = {}
    for p in preds:
        if p["score"] >= conf_thr:
            pred_by_img.setdefault(p["image_id"], []).append(p)
    tp = fp = fn = 0
    for img in images:
        truth = gt_by_img.get(img["id"], [])
        used = set()
        for p in sorted(pred_by_img.get(img["id"], []), key=lambda d: -d["score"]):
            best, best_j = 0.0, -1
            for j, g in enumerate(truth):
                if j in used or g["category_id"] != p["category_id"]:
                    continue
                v = _box_iou(p["bbox"], g["bbox"])
                if v > best:
                    best, best_j = v, j
            if best >= iou_thr:
                tp += 1
                used.add(best_j)
            else:
                fp += 1
        fn += len(truth) - len(used)
    return tp, fp, fn


def operating_point(images, annotations, preds, iou_thr: float = 0.5) -> dict:
    """Precision/recall/F1 at the confidence that maximizes F1 (point estimate only)."""
    scores = sorted({p["score"] for p in preds}, reverse=True)
    if len(scores) > 200:
        scores = scores[:: max(1, len(scores) // 200)]
    best = {"conf": 0.0, "precision": 0.0, "recall": 0.0, "f1": 0.0,
            "tp": 0, "fp": 0, "fn": len(annotations)}
    for thr in scores + [0.0]:
        tp, fp, fn = _counts_at(images, annotations, preds, thr, iou_thr)
        precision = tp / (tp + fp) if tp + fp else 0.0
        recall = tp / (tp + fn) if tp + fn else 0.0
        f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
        if f1 > best["f1"]:
            best = {"conf": float(thr), "precision": precision, "recall": recall,
                    "f1": f1, "tp": tp, "fp": fp, "fn": fn}
    return best


def evaluate(images, annotations, categories, preds, per_class=False) -> dict:
    from pycocotools.cocoeval import COCOeval
    out = {"mAP50": 0.0, "mAP50_95": 0.0, "iou_type": "segm"}
    if per_class:
        out["per_class_mAP50"] = {c["name"]: None for c in categories}
    if not preds or not annotations:
        return out
    gt = _coco_gt(images, annotations, categories)
    with contextlib.redirect_stdout(io.StringIO()):
        dt = gt.loadRes([dict(p) for p in preds])
        ev = COCOeval(gt, dt, "segm")
        ev.evaluate(); ev.accumulate(); ev.summarize()
    out["mAP50_95"], out["mAP50"] = float(ev.stats[0]), float(ev.stats[1])
    if per_class:
        present = {a["category_id"] for a in annotations}
        for cat in categories:
            if cat["id"] not in present:
                continue
            with contextlib.redirect_stdout(io.StringIO()):
                ev_c = COCOeval(gt, dt, "segm")
                ev_c.params.catIds = [cat["id"]]
                ev_c.evaluate(); ev_c.accumulate(); ev_c.summarize()
            out["per_class_mAP50"][cat["name"]] = float(ev_c.stats[1])
    return out


def _resample(images, annotations, preds_by_system, groups, draw_groups):
    imgs_of: dict[str, list[dict]] = {}
    for img in images:
        imgs_of.setdefault(groups[img["file_name"]], []).append(img)
    gt_by_img: dict[int, list[dict]] = {}
    for ann in annotations:
        gt_by_img.setdefault(ann["image_id"], []).append(ann)
    pred_by_img = {s: {} for s in preds_by_system}
    for system, preds in preds_by_system.items():
        for p in preds:
            pred_by_img[system].setdefault(p["image_id"], []).append(p)
    new_images, new_gt = [], []
    new_preds = {s: [] for s in preds_by_system}
    next_id, next_ann = 1, 1
    for group in draw_groups:
        for img in imgs_of.get(group, []):
            new_images.append({**img, "id": next_id})
            for ann in gt_by_img.get(img["id"], []):
                new_gt.append({**ann, "id": next_ann, "image_id": next_id})
                next_ann += 1
            for system in preds_by_system:
                for p in pred_by_img[system].get(img["id"], []):
                    new_preds[system].append({**p, "id": next_ann, "image_id": next_id})
                    next_ann += 1
            next_id += 1
    return new_images, new_gt, new_preds


def _ci(values):
    lo, hi = np.percentile(values, [2.5, 97.5])
    return [float(lo), float(hi)]


def compare(gt, preds_by_system, groups, pairs, n_boot: int, seed: int) -> dict:
    images, annotations, categories = gt["images"], gt["annotations"], gt["categories"]
    point = {s: evaluate(images, annotations, categories, p, per_class=True)
             for s, p in preds_by_system.items()}
    for s, p in preds_by_system.items():
        point[s]["operating_point_iou50"] = operating_point(images, annotations, p)
    result = {"n_images": len(images), "n_groups": len({groups[i["file_name"]] for i in images}),
              "n_bootstrap": n_boot, "systems": point, "differences": {}}
    if n_boot <= 0:
        return result
    group_list = sorted({groups[i["file_name"]] for i in images})
    rng = np.random.default_rng(seed)
    draws = {s: {"mAP50": [], "mAP50_95": []} for s in preds_by_system}
    for draw in range(n_boot):
        if draw % 25 == 0:
            _log(f"bootstrap draw {draw}/{n_boot}")
        chosen = list(rng.choice(group_list, size=len(group_list), replace=True))
        b_img, b_gt, b_pred = _resample(images, annotations, preds_by_system, groups, chosen)
        for system in preds_by_system:
            m = evaluate(b_img, b_gt, categories, b_pred[system])
            draws[system]["mAP50"].append(m["mAP50"])
            draws[system]["mAP50_95"].append(m["mAP50_95"])
    for system in preds_by_system:
        for metric in ("mAP50", "mAP50_95"):
            point[system][f"{metric}_ci95"] = _ci(draws[system][metric])
    for a, b in pairs:
        entry = {}
        for metric in ("mAP50", "mAP50_95"):
            diff = np.asarray(draws[a][metric]) - np.asarray(draws[b][metric])
            entry[metric] = {"point": point[a][metric] - point[b][metric], "ci95": _ci(diff),
                             "share_of_draws_above_zero": float((diff > 0).mean())}
        result["differences"][f"{LABELS[a]} minus {LABELS[b]}"] = entry
    return result


def render_markdown(res: dict, classes: list[str], note: str) -> str:
    fmt = lambda c: f"{c[0]:.4f} to {c[1]:.4f}" if c else "n/a"
    lines = [note, "", f"Images: {res['n_images']}, source groups: {res['n_groups']}, "
             f"bootstrap draws: {res['n_bootstrap']}", "",
             "| System | mAP50 | 95% CI | mAP50:95 | 95% CI |", "|---|---|---|---|---|"]
    for s, m in res["systems"].items():
        lines.append(f"| {LABELS[s]} | {m['mAP50']:.4f} | {fmt(m.get('mAP50_ci95'))} | "
                     f"{m['mAP50_95']:.4f} | {fmt(m.get('mAP50_95_ci95'))} |")
    lines += ["", "| System | " + " | ".join(classes) + " |", "|---|" + "---|" * len(classes)]
    for s, m in res["systems"].items():
        pc = m["per_class_mAP50"]
        lines.append(f"| {LABELS[s]} | " + " | ".join(
            "n/a" if pc[c] is None else f"{pc[c]:.4f}" for c in classes) + " |")
    lines += ["", "| System | Precision | Recall | F1 | at conf |",
              "|---|---|---|---|---|"]
    for s, m in res["systems"].items():
        op = m.get("operating_point_iou50") or {}
        lines.append(f"| {LABELS[s]} | {op.get('precision', 0):.4f} | {op.get('recall', 0):.4f} | "
                     f"{op.get('f1', 0):.4f} | {op.get('conf', 0):.3f} |")
    lines.append("\nPrecision/Recall/F1 are at the best-F1 confidence threshold, IoU 0.50.")
    if res["differences"]:
        lines += ["", "| Difference | mAP50 | 95% CI | mAP50:95 | 95% CI |", "|---|---|---|---|---|"]
        for label, e in res["differences"].items():
            a, b = e["mAP50"], e["mAP50_95"]
            lines.append(f"| {label} | {a['point']:+.4f} | {a['ci95'][0]:+.4f} to {a['ci95'][1]:+.4f} | "
                         f"{b['point']:+.4f} | {b['ci95'][0]:+.4f} to {b['ci95'][1]:+.4f} |")
    return "\n".join(lines) + "\n"


def _group_map(manifest: dict) -> dict[str, str]:
    by_name: dict[str, str] = {}
    for rel, group in manifest["group_of"].items():
        name = Path(rel).name
        if by_name.get(name, group) != group:
            raise ValueError(f"File name {name} maps to two source groups")
        by_name[name] = group
    return by_name


def run_stage_compare(args, manifest: dict, work: Path, selected: list[int]) -> dict:
    pdir = work / "predictions"
    out_dir = work / "results"
    out_dir.mkdir(parents=True, exist_ok=True)
    gt = json.loads((pdir / "oof_gt.json").read_text())
    systems = {}
    for cond_file in sorted(pdir.glob("detector_*.json")):
        systems[cond_file.stem.split("_", 1)[1]] = json.loads(cond_file.read_text())
    names = [c["name"] for c in gt["categories"]]
    pairs = [PRIMARY] if all(k in systems for k in PRIMARY) else []
    result = compare(gt, systems, _group_map(manifest), pairs, args.bootstrap, args.seed)
    result["primary_comparison"] = f"{LABELS[PRIMARY[0]]} minus {LABELS[PRIMARY[1]]}"
    note = ("Matched leakage experiment, YOLO12n-seg. Folds scored: " +
            ",".join(map(str, selected)) +
            ("" if len(selected) == 5 else "  (PARTIAL, not a five-fold result)") +
            ". Both conditions are evaluated on the same held-out images. Paired "
            f"difference ({result['primary_comparison']}) uses a bootstrap over "
            "source groups. Metrics are segmentation COCO AP.")
    (out_dir / "session39_comparison.json").write_text(json.dumps(result, indent=1))
    md = render_markdown(result, names, note)
    (out_dir / "session39_comparison.md").write_text(md)
    print(md, flush=True)
    return result


# ---------------------------------------------------------------- main

def build_parser() -> argparse.ArgumentParser:
    p = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    p.add_argument("--smoke", action="store_true",
                   help="fold 0, one epoch each, 20 bootstrap draws, separate work dir")
    p.add_argument("--data-root", default="auto")
    p.add_argument("--work-dir", default=None)
    p.add_argument("--folds", default="0,1,2,3,4")
    p.add_argument("--fold-seed", type=int, default=42)
    p.add_argument("--inner-frac", type=float, default=0.1,
                   help="share of each fold's training groups held out for model selection")
    p.add_argument("--model", default=DEFAULT_MODEL)  # ultralytics yaml or .pt
    p.add_argument("--init-weights", default="yolo12n.pt", help="transfer weights; 'none' trains from scratch")
    p.add_argument("--epochs", type=int, default=60)
    p.add_argument("--patience", type=int, default=0, help="unused; fixed schedule, final weights")
    p.add_argument("--imgsz", type=int, default=640)
    p.add_argument("--batch", type=int, default=8, help="per GPU; halves itself on out-of-memory")
    p.add_argument("--workers", type=int, default=2)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--conf", type=float, default=0.001)
    p.add_argument("--iou", type=float, default=0.7)
    p.add_argument("--det-max", type=int, default=100)
    p.add_argument("--bootstrap", type=int, default=200)
    p.add_argument("--total-budget-hours", type=float, default=11.0)
    p.add_argument("--reserve-hours", type=float, default=0.5,
                   help="held back at the end of the session for scoring")
    p.add_argument("--resume-from", action="append", default=[])
    p.add_argument("--plan-only", action="store_true")
    p.add_argument("--worker", default=None, help=argparse.SUPPRESS)
    p.add_argument("--fold", type=int, default=None, help=argparse.SUPPRESS)
    p.add_argument("--conditions", default="grouped")
    p.add_argument("--condition", default=None, help=argparse.SUPPRESS)
    return p


def _worker_main(args) -> int:
    work = Path(args.work_dir)
    run = json.loads((work / "run_args.json").read_text())
    plan = json.loads((work / "plan.json").read_text())
    run["condition"] = args.condition
    return worker_det(run, plan, args.fold, work)


def main(argv=None) -> int:
    args, _ = build_parser().parse_known_args(argv)
    if args.worker:
        return _worker_main(args)
    if args.smoke:
        args.folds, args.epochs, args.bootstrap = "0", 1, 20
        args.work_dir = args.work_dir or "/kaggle/working/smoke39"
    started = time.time()
    deadline = started + (args.total_budget_hours - args.reserve_hours) * 3600
    self_path = _self_script()
    work = Path(args.work_dir) if args.work_dir else (
        Path("/kaggle/working") / SESSION_NAME if Path("/kaggle/working").is_dir()
        else Path.cwd() / SESSION_NAME)
    work.mkdir(parents=True, exist_ok=True)
    selected = sorted({int(f) for f in args.folds.split(",") if f.strip()})

    cv_data = load_dataset(args.data_root)
    names = cv_data["names"]
    manifest = fold_manifest(cv_data, 5, args.fold_seed)
    folds = make_folds(cv_data["trainval"], len(names), 5, args.fold_seed)
    for fold in manifest["folds"]:
        if {manifest["group_of"][p] for p in fold["valid"]} & \
                {manifest["group_of"][p] for p in fold["train"]}:
            raise AssertionError(f"Fold {fold['fold']} leaks a source group")
    (work / "folds.json").write_text(json.dumps(manifest, indent=1))
    (work / "plan_signature.txt").write_text(manifest["signature"])
    _log(f"fold assignment rebuilt, signature={manifest['signature'][:16]}")

    for src in args.resume_from:
        import_previous(Path(src), work, manifest["signature"])
    plan = build_plan(cv_data, folds, selected, work, args.inner_frac, args.seed)
    (work / "plan.json").write_text(json.dumps(plan))
    if args.plan_only:
        return 0

    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet",
                           "ultralytics", "pycocotools"])
    import torch
    gpus = list(range(torch.cuda.device_count()))
    if not gpus:
        raise RuntimeError("No GPU found; enable GPU T4 x2 in the Kaggle settings")
    run = {k: getattr(args, k) for k in (
        "model", "init_weights", "epochs", "patience", "imgsz", "batch", "workers", "seed",
        "conf", "iou", "det_max", "fold_seed", "data_root")}
    conditions = [c.strip() for c in args.conditions.split(",") if c.strip()]
    if not conditions or any(c not in CONDITIONS for c in conditions):
        raise ValueError(f"--conditions needs names from {CONDITIONS}")
    run.update(signature=manifest["signature"], folds=selected, conditions=conditions)
    (work / "run_args.json").write_text(json.dumps(run, indent=1))
    base = [sys.executable, "-u", str(self_path), "--work-dir", str(work)]

    det_jobs = [{"name": f"detector_fold_{k}_{c}",
                 "cmd": base + ["--worker", "det", "--condition", c, "--fold", str(k)],
                 "done": (lambda k=k, c=c: (work / "det" / c / f"det_fold_{k}.json").is_file()),
                 "status": (lambda k=k, c=c: f"{_epochs_done(work / 'runs' / f'fold_{k}_{c}')} "
                                           f"of {args.epochs} epochs")}
                for c in conditions for k in selected]
    _log(f"GPUs: {gpus}. {len(det_jobs)} fold-condition jobs, one per GPU.")
    complete = run_jobs(det_jobs, gpus, deadline, work / "logs")
    if not complete:
        done = [f"{k}_{c}" for c in conditions for k in selected
                if (work / "det" / c / f"det_fold_{k}.json").is_file()]
        _log(f"session budget used. Detector folds finished: {done} of {selected}. Start a new "
             "session with the same cell, attach this notebook's output and set "
             "RUN_ARGS = ['--resume-from', '/kaggle/input/<this-output>'].")
        return 1

    if not (work / "predictions" / "oof_gt.json").is_file():
        run_stage_collect(run, work)

    run_stage_compare(args, manifest, work, selected)
    zip_path = work / "SESSION_39_results.zip"
    with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
        for sub in ("results", "det", "predictions"):
            for f in sorted((work / sub).glob("*.json")) + sorted((work / sub).glob("*.md")):
                zf.write(f, arcname=f"{sub}/{f.name}")
        for csv_file in sorted((work / "runs").glob("fold_*/results.csv")):
            zf.write(csv_file, arcname=f"detector_logs/{csv_file.parent.name}_results.csv")
        zf.write(work / "folds.json", arcname="folds.json")
    _log(f"done in {(time.time() - started) / 3600:.2f} h. Results: {zip_path}")
    return 0


if __name__ == "__main__":
    sys.exit(main(RUN_ARGS if (RUN_ARGS and not sys.argv[1:]) else None))


  [data] selected dataset: /kaggle/input/datasets/muntasirulmunna/image-annotation-rebuilt
  [data] filename-derived source groups: 310 groups, 8022/8172 images kept (6 groups excluded for class-set disagreement); 310 groups span more than one split -- all splits pooled for grouped CV
  [data] root=/kaggle/input/datasets/muntasirulmunna/image-annotation-rebuilt  classes=4  cv_images=8022
  [data] class dist: {'acne': 1921, 'Nail_psoriasis': 7866, 'vitiligo': 2779, 'SJS-TEN': 3417}
[39] 19:11:55 fold assignment rebuilt, signature=8e7a6e2d325d4fb3
[39] 19:18:08 fold 0: fit=5796 images (+1524 overlap siblings, 62 canonical/pixel-dup excluded), inner val=640 images (25 groups), held-out=61
[39] 19:24:21 fold 1: fit=5710 images (+1562 overlap siblings, 71 canonical/pixel-dup excluded), inner val=679 images (25 groups), held-out=64
[39] 19:30:28 fold 2: fit=5768 images (+1537 overlap siblings, 69 canonical/pixel-dup excluded), inner val=648 images (25 groups), held-out=62
[39] 19:36:30 fold 

SystemExit: 0

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3561: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
